In [ ]:
# https://gemini.google.com/app/5109b1ed662b462a


import os
import subprocess
try:
    _printenv = subprocess.run(
        ['bash', '-c', 'source ~/.bashrc 2>/dev/null && printenv'],
        text=True, capture_output=True, timeout=10,
    ).stdout
    for _line in _printenv.splitlines():
        if '=' in _line:
            _k, _v = _line.split('=', 1)
            os.environ.setdefault(_k, _v)
except Exception:
    pass
if 'PDK_ROOT' in os.environ and 'PDK' in os.environ:
    os.environ.setdefault('PDKPATH', os.path.join(os.environ['PDK_ROOT'], os.environ['PDK']))

In [ ]:
import gdstk
import svgutils.transform as sg
import IPython.display
from IPython.display import clear_output
import ipywidgets as widgets

# Redirect all outputs here
hide = widgets.Output()

def display_gds(gds_file,path,scale = 3):
  
  # Generate an SVG image
  top_level_cell = gdstk.read_gds(gds_file).top_level()[0]
  top_level_cell.write_svg(os.path.join(path,'out.svg'))
    
  # Scale the image for displaying
  fig = sg.fromfile(os.path.join(path,'out.svg'))
  fig.set_size((str(float(fig.width) * scale), str(float(fig.height) * scale)))
  fig.save(os.path.join(path,'out.svg'))

  # Display the image
  IPython.display.display(IPython.display.SVG(os.path.join(path,'out.svg')))
  os.remove(os.path.join(path,'out.gds'))

def display_component(component,path,scale = 3):
  # Save to a GDS file
  with hide:
    component.write_gds(os.path.join(path,'out.gds'))
  display_gds(os.path.join(path,'out.gds'),path,scale)

In [ ]:
# %%
import os
import gdsfactory as gf
from gdsfactory import Component
from glayout import MappedPDK, gf180
from glayout import nmos, pmos
from glayout.primitives.mimcap import mimcap_array
from glayout.routing.straight_route import straight_route
from glayout.routing.c_route import c_route
from glayout.routing.L_route import L_route
from glayout.util.comp_utils import align_comp_to_port, evaluate_bbox

In [ ]:

# Base parameters for PMOS and NMOS
stdp_config = {
    "pdk": gf180,
    "layout_rules": {
        "spacing": gf180.util_max_metal_seperation(),
        "routing_metal": "met2",
        "dummy_devices": False,
        "tie_layers": ("met2", "met1"),
        "sd_rmult": 1,
    },
}

nmos_kwargs = {
    "with_tie": True,
    "with_dnwell": True,
    "sd_route_topmet": "met2",
    "gate_route_topmet": "met2",
    "sd_route_left": True,
    "rmult": None,
    "gate_rmult": 1,
    "interfinger_rmult": 1,
    "substrate_tap_layers": ("met2","met1"),
    "dummy_routes": True
}

pmos_kwargs = {
    "with_tie": True,
    "dnwell": False,
    "sd_route_topmet": "met2",
    "gate_route_topmet": "met2",
    "sd_route_left": True,
    "rmult": None,
    "gate_rmult": 1,
    "interfinger_rmult": 1,
    "substrate_tap_layers": ("met2","met1"),
    "dummy_routes": True
}

In [ ]:

pdk = gf180
stdp_comp = Component(name="stdp_lvs")
rules = stdp_config["layout_rules"]
spacing = rules["spacing"]
tie_layers = rules["tie_layers"]
sd_rmult = rules["sd_rmult"]

# =========================================================================
# 1. INSTANTIATE ALL DEVICES MATCHING SPICE NETLIST
# =========================================================================

# --- PMOS TRANSISTORS ---
# XM1: W=0.22u, L=0.28u
xm1 = pmos(pdk, width=0.5, length=0.28, with_dummy=(False, False), with_substrate_tap=True, tie_layers=tie_layers, sd_rmult=sd_rmult, **pmos_kwargs)
# XM2: W=0.22u, L=0.28u
xm2 = pmos(pdk, width=0.5, length=0.28, with_dummy=(False, False), with_substrate_tap=True, tie_layers=tie_layers, sd_rmult=sd_rmult, **pmos_kwargs)
# XM7: W=0.22u, L=0.28u
xm7 = pmos(pdk, width=0.5, length=0.28, with_dummy=(False, False), with_substrate_tap=True, tie_layers=tie_layers, sd_rmult=sd_rmult, **pmos_kwargs)
# XM8: W=0.22u, L=0.28u
xm8 = pmos(pdk, width=0.5, length=0.28, with_dummy=(False, False), with_substrate_tap=True, tie_layers=tie_layers, sd_rmult=sd_rmult, **pmos_kwargs)
# XM17: W=0.3u, L=4.6u
xm17 = pmos(pdk, width=0.5, length=4.6, with_dummy=(False, False), with_substrate_tap=True, tie_layers=tie_layers, sd_rmult=sd_rmult, **pmos_kwargs)
# XM16: W=10.2u, L=2.8u
xm16 = pmos(pdk, width=10.2, length=2.8, with_dummy=(False, False), with_substrate_tap=True, tie_layers=tie_layers, sd_rmult=sd_rmult, **pmos_kwargs)

# --- NMOS TRANSISTORS ---
# XM3: W=0.22u, L=0.28u
xm3 = nmos(pdk, width=0.5, length=0.28, with_dummy=(False, False), with_substrate_tap=True, tie_layers=tie_layers, sd_rmult=sd_rmult, **nmos_kwargs)
# XM4: W=0.22u, L=0.28u
xm4 = nmos(pdk, width=0.5, length=0.28, with_dummy=(False, False), with_substrate_tap=True, tie_layers=tie_layers, sd_rmult=sd_rmult, **nmos_kwargs)
# XM9: W=0.22u, L=0.28u
xm9 = nmos(pdk, width=0.5, length=0.28, with_dummy=(False, False), with_substrate_tap=True, tie_layers=tie_layers, sd_rmult=sd_rmult, **nmos_kwargs)
# XM12: W=0.22u, L=0.28u
xm12 = nmos(pdk, width=0.5, length=0.28, with_dummy=(False, False), with_substrate_tap=True, tie_layers=tie_layers, sd_rmult=sd_rmult, **nmos_kwargs)
# XMCM_1: W=0.61u, L=2.8u
xmcm_1 = nmos(pdk, width=0.61, length=2.8, with_dummy=(False, False), with_substrate_tap=True, tie_layers=tie_layers, sd_rmult=sd_rmult, **nmos_kwargs)

# --- LONG-CHANNEL BIAS CHAIN (XMCM_5 to XMCM_9): W=0.5u, L=10u ---
xmcm_chain = [
    nmos(pdk, width=0.5, length=10.0, with_dummy=(False, False), with_substrate_tap=True, tie_layers=tie_layers, sd_rmult=sd_rmult, **nmos_kwargs)
    for _ in range(5)
]

# --- MIM CAPACITORS ---
# Cpot: 5x5um, m=2 (2 rows, 1 col)
cpot = mimcap_array(pdk, size=(5, 5), rows=2, columns=1)
# Cdep: 5x5um, m=2 (2 rows, 1 col)
cdep = mimcap_array(pdk, size=(5, 5), rows=2, columns=1)
# Cw: 5x5um, m=10 (2 rows, 5 cols)
cw = mimcap_array(pdk, size=(5, 5), rows=5, columns=2)

# =========================================================================
# 2. PLACEMENT & FLOORPLANNING
# =========================================================================

# Add components to subcircuit
#top row
xm1_ref = stdp_comp << xm1
xm2_ref = stdp_comp << xm2
xm7_ref = stdp_comp << xm7
xm8_ref = stdp_comp << xm8
xm17_ref = stdp_comp << xm17
xm16_ref = stdp_comp << xm16


# second row
xm3_ref = stdp_comp << xm3
xm4_ref = stdp_comp << xm4
xm9_ref = stdp_comp << xm9
xm12_ref = stdp_comp << xm12
xmcm_1_ref = stdp_comp << xmcm_1
chain_refs = [stdp_comp << dev for dev in xmcm_chain]

cpot_ref = stdp_comp << cpot
cdep_ref = stdp_comp << cdep
cw_ref = stdp_comp << cw

xm1_ref.name = "xm1" 
xm2_ref.name = "xm2" 
xm7_ref.name = "xm7" 
xm8_ref.name = "xm8" 
xm17_ref.name = "xm17" 
xm16_ref.name = "xm16" 
xm3_ref.name = "xm3"
xm4_ref.name = "xm4"
xm9_ref.name = "xm9"
xm12_ref.name = "xm12"
cpot_ref.name = "cpot" 
cdep_ref.name = "cdep" 
cw_ref.name = "cw" 

#placement of three rows
top_row_y = 40
middle_row_y = 20
bottom_row_y = 0
# --- TOP ROW: PMOS Array ---
xm1_xy = evaluate_bbox(xm1)
xm2_xy = evaluate_bbox(xm2)
xm7_xy = evaluate_bbox(xm7)
xm8_xy = evaluate_bbox(xm8)
xm17_xy = evaluate_bbox(xm17)
xm16_xy = evaluate_bbox(xm16)
cpot_xy = evaluate_bbox(cpot)
xm1_ref.move((0 + xm1_xy[0]/2, top_row_y))
xm2_ref.move((xm1_ref.xmax + xm2_xy[0]/2 + spacing, top_row_y))
xm7_ref.move((xm2_ref.xmax + xm7_xy[0]/2 + spacing, top_row_y))
xm8_ref.move((xm7_ref.xmax + xm8_xy[0]/2 + spacing, top_row_y))
xm17_ref.move((xm8_ref.xmax + xm17_xy[0]/2 + spacing, top_row_y))
xm16_ref.move((xm17_ref.xmax + xm16_xy[0]/2 + spacing, top_row_y))


# --- MIDDLE ROW: Standard NMOS Array ---
xm3_xy = evaluate_bbox(xm3)
xm4_xy = evaluate_bbox(xm4)
xm9_xy = evaluate_bbox(xm9)
xm12_xy = evaluate_bbox(xm12)
xmcm_1_xy = evaluate_bbox(xmcm_1)
cdep_xy = evaluate_bbox(cdep)

xm3_ref.move((0 + xm3_xy[0]/2, middle_row_y))
xm4_ref.move((xm3_ref.xmax + xm4_xy[0]/2 + spacing, middle_row_y))
xm9_ref.move((xm4_ref.xmax + xm9_xy[0]/2 +spacing, middle_row_y))
xm12_ref.move((xm9_ref.xmax + xm12_xy[0]/2 + spacing, middle_row_y))
xmcm_1_ref.move((xm12_ref.xmax + xmcm_1_xy[0]/2 + spacing, middle_row_y))


# --- BOTTOM ROW: Long-Channel Bias Chain (Series connected) ---
chain_refs[0].move((xmcm_1_ref.xmax + evaluate_bbox(chain_refs[0])[0]/2 + spacing, middle_row_y))
chain_refs[1].move((evaluate_bbox(chain_refs[1])[0]/2, bottom_row_y))
chain_refs[2].move((chain_refs[1].xmax + evaluate_bbox(chain_refs[2])[0]/2 + spacing, bottom_row_y))
chain_refs[3].move((chain_refs[2].xmax + evaluate_bbox(chain_refs[3])[0]/2 + spacing, bottom_row_y))
chain_refs[4].move((chain_refs[3].xmax + evaluate_bbox(chain_refs[4])[0]/2 + spacing, bottom_row_y))

# xy_first = evaluate_bbox(chain_refs[0])
# curr_x = xy_first[0]/2
# for ref in chain_refs:
#     ref.move((curr_x , 0))
#     xy_next = evaluate_bbox(ref)
#     curr_x = ref.xmax + xy_next[0]/2 + spacing

# chain_refs[4].move((-90, 10))

# --- MOVING CAPACITORS: 
cw_xy = evaluate_bbox(cw)
cpot_ref.move((xm16_ref.xmax + cpot_xy[0]/2 + spacing * 2, middle_row_y))
cdep_ref.move((cpot_ref.xmax + cdep_xy[0]/2 + spacing * 2, middle_row_y))
cw_ref.move((cdep_ref.xmax + cdep_xy[0]/2 + spacing * 2, bottom_row_y))




In [ ]:
stdp_comp.show()


In [ ]:
# --- Internal Net 1: XM1 Source -> XM2 Drain ---
stdp_comp << straight_route(
    pdk, 
    xm1_ref.ports["multiplier_0_source_E"], 
    xm2_ref.ports["multiplier_0_drain_W"]
)

# --- Internal Net 2: XM3 Source -> XM4 Drain ---
stdp_comp << straight_route(
    pdk, 
    xm3_ref.ports["multiplier_0_source_E"], 
    xm4_ref.ports["multiplier_0_drain_W"]
)

# # --- Internal Net 3: XM7 Drain -> XMCM_1 Drain ---
stdp_comp << c_route(
    pdk, 
    xm7_ref.ports["multiplier_0_drain_N"], 
    xmcm_1_ref.ports["multiplier_0_drain_N"],
    extension=1.5
)

# --- Internal Net 4: XM7 Source -> XM8 Drain ---
stdp_comp << straight_route(
    pdk, 
    xm7_ref.ports["multiplier_0_source_E"], 
    xm8_ref.ports["multiplier_0_drain_W"]
)

# # --- Internal Net 5: Diode Load XM9 (Gate to Drain) ---
stdp_comp << c_route(
    pdk, 
    xm9_ref.ports["multiplier_0_gate_W"], 
    xm9_ref.ports["multiplier_0_drain_W"],
    extension=0.8
)

# --- Internal Net 5 Node: XM9 Drain -> XM12 Drain & XM16 Drain ---
stdp_comp << straight_route(
    pdk, 
    xm9_ref.ports["multiplier_0_drain_E"], 
    xm12_ref.ports["multiplier_0_drain_W"]
)
stdp_comp << c_route(
    pdk, 
    xm12_ref.ports["multiplier_0_drain_E"], 
    xm16_ref.ports["multiplier_0_drain_E"],
    extension=2.5
)

# --- Bias Chain Series Connections (Net 6, Net 7, Net 8, Net 9) ---
# XMCM_5 Source -> XMCM_6 Drain
stdp_comp << straight_route(pdk, chain_refs[0].ports["multiplier_0_source_E"], chain_refs[1].ports["multiplier_0_drain_W"])
# XMCM_6 Source -> XMCM_7 Drain
stdp_comp << straight_route(pdk, chain_refs[1].ports["multiplier_0_source_E"], chain_refs[2].ports["multiplier_0_drain_W"])
# XMCM_7 Source -> XMCM_9 Drain
stdp_comp << straight_route(pdk, chain_refs[2].ports["multiplier_0_source_E"], chain_refs[4].ports["multiplier_0_drain_W"])
# XMCM_9 Source -> XMCM_8 Drain
stdp_comp << straight_route(pdk, chain_refs[4].ports["multiplier_0_source_E"], chain_refs[3].ports["multiplier_0_drain_W"])

# # --- Common Gates for Bias Chain (vb_itd) ---
for i in range(4):
    stdp_comp << straight_route(pdk, chain_refs[i].ports["multiplier_0_gate_E"], chain_refs[i+1].ports["multiplier_0_gate_W"])



In [ ]:
# --- Nodes connected to Passive Capacitors ---
# vpot Node: XM1 Gate -> XM17 Drain -> Cpot Bottom Metal
stdp_comp << straight_route(pdk, xm1_ref.ports["multiplier_0_gate_E"], xm17_ref.ports["multiplier_0_drain_W"])
stdp_comp << c_route(pdk, xm17_ref.ports["multiplier_0_drain_E"], cpot_ref.ports["row0_col0_bottom_met_E"], extension=2.0)

# vdep Node: XM4 Gate -> XM12 Source -> Cdep Top Metal -> XMCM_5 Drain
stdp_comp << straight_route(pdk, xm4_ref.ports["multiplier_0_gate_E"], xm12_ref.ports["multiplier_0_source_W"])
stdp_comp << c_route(pdk, xm12_ref.ports["multiplier_0_source_W"], cdep_ref.ports["row0_col0_top_met_W"], extension=1.5)
stdp_comp << L_route(pdk, cdep_ref.ports["row1_col0_top_met_S"], chain_refs[0].ports["multiplier_0_drain_E"])

# vw Node: XM2 Drain -> XM3 Drain -> Cw Top Metal
stdp_comp << c_route(pdk, xm2_ref.ports["multiplier_0_drain_W"], xm3_ref.ports["multiplier_0_drain_W"], extension=1.0)
stdp_comp << c_route(pdk, xm3_ref.ports["multiplier_0_drain_W"], cw_ref.ports["row0_col0_top_met_W"], extension=3.0)

In [ ]:
display_component(stdp_comp, scale = 1,path=".")

In [ ]:
# =========================================================================
# 4. ADD PORTS TO THE COMPONENT (MATCHING SPICE PINLIST)
# =========================================================================

# --- Power & Ground ---
stdp_comp.add_port(
    "avdd", port=xm1_ref.ports["multiplier_0_source_W"]
)
stdp_comp.add_port(
    "avss", port=xm3_ref.ports["multiplier_0_source_W"]
)

# --- Control / Input Pins (Transistor Gates) ---
stdp_comp.add_port("nvpost", port=xm2_ref.ports["multiplier_0_gate_E"])
stdp_comp.add_port("nvpre", port=xm8_ref.ports["multiplier_0_gate_E"])
stdp_comp.add_port("vpost", port=xm12_ref.ports["multiplier_0_gate_E"])
stdp_comp.add_port("vpre", port=xm3_ref.ports["multiplier_0_gate_E"])

# --- Bias Voltages ---
stdp_comp.add_port("vb_idep", port=xm16_ref.ports["multiplier_0_gate_E"])
stdp_comp.add_port(
    "vb_itd", port=chain_refs[0].ports["multiplier_0_gate_W"]
)  # Bias chain tail
stdp_comp.add_port("vb_itp", port=xm17_ref.ports["multiplier_0_gate_E"])
stdp_comp.add_port("vb_pot", port=xmcm_1_ref.ports["multiplier_0_gate_E"])

# --- Output Node ---
stdp_comp.add_port("vw", port=cw_ref.ports["row0_col0_top_met_E"])



In [ ]:
# =========================================================================
# 5. ADD MET2 PHYSICAL PINS AND LABELS FOR LVS
# =========================================================================

# Define MET2 pin and label layers for GF180MCU
MET2_PIN_LAYER = (38, 5)  # met2.pin
MET2_LABEL_LAYER = (38, 5)  # met2.label / met2.pin

# Map port names to their source port objects
pins_to_create = {
    "avdd": xm1_ref.ports["multiplier_0_source_W"],
    "avss": xm3_ref.ports["multiplier_0_source_W"],
    "nvpost": xm2_ref.ports["multiplier_0_gate_E"],
    "nvpre": xm8_ref.ports["multiplier_0_gate_E"],
    "vpost": xm12_ref.ports["multiplier_0_gate_E"],
    "vpre": xm3_ref.ports["multiplier_0_gate_E"],
    "vb_idep": xm16_ref.ports["multiplier_0_gate_E"],
    "vb_itd": chain_refs[0].ports["multiplier_0_gate_W"],
    "vb_itp": xm17_ref.ports["multiplier_0_gate_E"],
    "vb_pot": xmcm_1_ref.ports["multiplier_0_gate_E"],
    "vw": cw_ref.ports["row0_col0_top_met_E"],
}

for pin_name, port in pins_to_create.items():
  # Extract center position, size, and orientation from port object
  center = port.center
  width = port.width
  height = port.width  # Standard square pin area around the port center

  # 1. Add pin rectangle shape on met2_pin
  stdp_comp.add_polygon(
      [
          (center[0] - width / 2, center[1] - height / 2),
          (center[0] + width / 2, center[1] - height / 2),
          (center[0] + width / 2, center[1] + height / 2),
          (center[0] - width / 2, center[1] + height / 2),
      ],
      layer=MET2_PIN_LAYER,
  )

  # 2. Add text label on met2_pin layer for net identity
  stdp_comp.add_label(
      text=pin_name, position=center, layer=MET2_LABEL_LAYER, magnification=0.3
  )

# # Save final layout with physical pins & labels
# stdp_comp.write_gds("stdp_lvs_complete.gds")

In [ ]:
display_component(stdp_comp, scale = 1,path=".")

In [ ]:
stdp_comp.show()
